# skyhunt — sesje zdjęć RAW (Run all)

Każdy podfolder `MyDrive/skyhunt/raw/<sesja>/` z plikami RAF to jedna sesja. Miejsce obserwacji i gwiazda-podpowiedź są w `MyDrive/skyhunt/sites.yaml` (wpisuje je komórka „Nagrania” w `run_skyhunt.ipynb`) — ten notebook ich nie zawiera.

**Środowisko wykonawcze → Uruchom wszystko.** Gotowe etapy są pomijane (`manifest.json` na Drive). Po nowym runtime etap `streaks` dekoduje RAF od nowa (~35 min na 900 zdjęć); stos i astrometria zostają z Drive.

`FIRST` w komórce ścieżek: ta sesja liczy się pierwsza (szybki wynik do oceny).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob

REPO_URL = 'https://github.com/LukaszCiesielski-lodz/skytrack_uap.git'
BRANCH = 'main'
REPO = '/content/skytrack_uap'
DRIVE_ROOT = '/content/drive/MyDrive/skyhunt'
RAW = f'{DRIVE_ROOT}/raw'
OUT = f'{DRIVE_ROOT}/out'
CONFIG = f'{DRIVE_ROOT}/config.yaml' if os.path.exists(f'{DRIVE_ROOT}/config.yaml') else f'{REPO}/config.yaml'
FIRST = 's2_deneb_0210'          # ta sesja pierwsza; '' = kolejność alfabetyczna
SESSIONS = None                  # None = wszystkie podfoldery z RAF; albo lista nazw, np. ['s2_deneb_0210']
COPY_SITE_FROM = 's2_deneb_0210'  # sesje bez wpisu w sites.yaml dostają miejsce tej sesji (bez współrzędnych w notebooku)

os.environ['SKYHUNT_SITES'] = f'{DRIVE_ROOT}/sites.yaml'   # miejsce obserwacji (poza repo)
found = sorted(d for d in os.listdir(RAW) if os.path.isdir(f'{RAW}/{d}')
               and len(glob.glob(f'{RAW}/{d}/*.[rR][aA][fF]')) >= 3)
SESSIONS = SESSIONS or sorted(found, key=lambda d: (d != FIRST, d))
import yaml
sites = (yaml.safe_load(open(os.environ['SKYHUNT_SITES'])) if os.path.exists(os.environ['SKYHUNT_SITES']) else None) or {}
src = sites.get(COPY_SITE_FROM) or {}
added = [d for d in SESSIONS if 'dark' not in d.lower() and 'lat_deg' not in (sites.get(d) or {}) and 'lat_deg' in src]
for d in added:
    sites[d] = {**(sites.get(d) or {}), **{k: src[k] for k in ('lat_deg', 'lon_deg', 'elevation_m')}}
if added:
    with open(os.environ['SKYHUNT_SITES'], 'w') as fh:
        yaml.safe_dump(sites, fh, allow_unicode=True)
    print(f'miejsce obserwacji skopiowane z {COPY_SITE_FROM}: {added}')
print('config:', CONFIG)
print('sites.yaml:', 'jest' if os.path.exists(os.environ['SKYHUNT_SITES']) else 'BRAK — uruchom komórkę „Nagrania” w run_skyhunt.ipynb')
for d in SESSIONS:
    print(f'{d:24s} {len(glob.glob(f"{RAW}/{d}/*.[rR][aA][fF]")):5d} RAF')

In [ ]:
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --oneline

In [ ]:
!pip install -q -e "{REPO}[gpu,astro,report,dev,raw]"
# astrometry.net: plate solve nowych sesji; poppler: podgląd stron PDF w ostatniej komórce
!apt-get -qq update > /dev/null && apt-get -qq install -y astrometry.net poppler-utils > /dev/null && which solve-field pdftoppm
import torch
print('torch', torch.__version__, 'GPU', torch.cuda.is_available())

In [ ]:
# Space-Track z Colab Secrets (SPACETRACK_USER, SPACETRACK_PASSWORD); kod ich nie wypisuje
try:
    from google.colab import userdata
    for key in ('SPACETRACK_USER', 'SPACETRACK_PASSWORD'):
        os.environ[key] = userdata.get(key)
    print('Space-Track: dane logowania ustawione')
except Exception:
    print('Space-Track: brak danych w Secrets — tylko CelesTrak')

In [ ]:
# testy (bez prawdziwego nagrania wideo: te testy będą pominięte)
os.environ['SKYHUNT_TEST_VIDEO'] = ''
!cd {REPO} && python -m pytest -q -rs 2>&1 | tail -n 15

In [ ]:
# sesje po kolei; pełny log → Drive (skyhunt_photos.log), w komórce powtórzenia skrócone
LOG = f'{OUT}/skyhunt_photos.log'
for s in SESSIONS:
    print(f'\n######## {s}')
    !set -o pipefail; skyhunt run "{RAW}/{s}" --config "{CONFIG}" --out "{OUT}" 2>&1 | tee -a "{LOG}" | awk -f "{REPO}/colab/dedup_log.awk"

In [ ]:
import json, subprocess
from IPython.display import Image, display

def fmt(v, spec):
    return format(v, spec) if isinstance(v, (int, float)) else '–'

for s in SESSIONS:
    o = f'{OUT}/{s}'
    print(f'\n======== {s}')
    !skyhunt status "{RAW}/{s}" --config "{CONFIG}" --out "{OUT}"
    if os.path.exists(f'{o}/link.json'):
        lk = json.load(open(f'{o}/link.json'))
        t = lk['timing']
        print(f"łańcuchy {lk['n_chains']} (najdłuższe {lk['chain_lengths'][:8]}), pojedyncze {lk['n_single']}")
        print(f"przerwa g = {fmt(t['g_s'], '.3f')} ± {fmt(t['sigma_s'], '.3f')} s, residua {fmt(t['rms_px'], '.2f')} px = {fmt(t['rms_ms'], '.0f')} ms, "
              f"rozrzut startu serii {fmt(t.get('set_jitter_ms'), '.0f')} ms ({t.get('set_jitter_pairs', 0)} par)")
    if os.path.exists(f'{o}/time_sync.json'):
        ts = json.load(open(f'{o}/time_sync.json'))
        chk = (ts.get('photo_timing') or {}).get('satellite_check') or {}
        print(f"Δ = {fmt(ts['delta_s'], '+.3f')} ± {fmt(ts['sigma_s'], '.3f')} s ({ts['confidence']}); kontrola z satelitów: "
              f"{chk.get('n_tracks', 0)} torów, residua {fmt(chk.get('rms_ms'), '.0f')} ms")

# podgląd raportu pierwszej sesji: mapa, rogi, przebieg, kolor, czas, krótkie obiekty, tabela
pdf = f'{OUT}/{SESSIONS[0]}/report/summary.pdf'
if os.path.exists(pdf):
    prev = '/content/summary_preview'
    subprocess.run(['rm', '-rf', prev]); os.makedirs(prev)
    subprocess.run(['pdftoppm', '-r', '70', '-png', '-f', '1', '-l', '8', pdf, f'{prev}/p'], check=False)
    for p in sorted(glob.glob(f'{prev}/p*.png')):
        display(Image(p, width=1100))